# 🧲 Classification de Défauts de Surface — Magnetic Tile Defect Dataset

Ce notebook traite la **classification supervisée** des défauts de surface de tuiles magnétiques
(*magnetic tiles*) — un dataset industriel **hidden gem** beaucoup moins exploité que NEU-DET.

> **Dataset**: [`alex000kim/magnetic-tile-surface-defects`](https://www.kaggle.com/datasets/alex000kim/magnetic-tile-surface-defects)
> **Référence académique**: Huang et al., *"Surface Defect Saliency of Magnetic Tile"*, The Visual Computer (2018).

### Pourquoi ce dataset ?

| Critère | Magnetic Tile | NEU-DET |
|---|---|---|
| Type d'image | Niveaux de gris industriels | Niveaux de gris industriels |
| Nb classes | **6** (Blowhole, Break, Crack, Fray, Uneven, Free) | 6 (Crazing, Inclusion, …) |
| Nb total d'images | ~1344 | 1800 |
| **Masques pixel-précis** | **✅ Oui** (PNG binaires) | ❌ Non (juste BBox XML) |
| **Déséquilibre de classes** | **✅ Sévère** (32 → 952) | ❌ Parfaitement équilibré |
| Popularité Kaggle | Faible (hidden gem) | Très élevé |

### Pipeline (et améliorations vs. NEU-DET)

1. Chargement récursif (`MT_<class>/Imgs/`) + lecture des **masques associés**
2. EDA + analyse de la distribution des tailles d'images (très variable ici)
3. Prétraitement : grayscale `224×224`, normalisation, **CLAHE**
4. Split **stratifié 70/15/15** sur le dataset complet
5. **Gestion du déséquilibre** : `class_weight` + **Focal Loss** (γ=2)
6. Augmentation industrielle + **CutMix** (upgrade de Mixup)
7. **CNN custom from scratch** à 4 blocs (baseline)
8. **CNN amélioré** : blocs Résiduels + **SE + CBAM** (attention spatiale + canal)
9. **Cosine LR avec warmup**
10. **Transfer Learning** EfficientNetV2B0 en 2 stages
11. **TTA** ×10
12. **Calibration** : reliability diagram + ECE
13. **Grad-CAM** + **mesure IoU avec les masques GT** (innovation unique de ce dataset !)
14. t-SNE + UMAP des embeddings
15. **Tests de robustesse** (bruit, flou, luminosité)
16. Tableau comparatif final

---


# 1. Importation des bibliothèques

In [ ]:
import os
import re
import math
import random
import time
import warnings
import json
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

warnings.filterwarnings("ignore")

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.utils import to_categorical

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, accuracy_score,
    f1_score, precision_recall_fscore_support, roc_auc_score
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.manifold import TSNE
from sklearn.metrics.pairwise import cosine_similarity

from tqdm import tqdm

# Reproductibilité
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# GPU
gpus = tf.config.list_physical_devices("GPU")
if gpus:
    for g in gpus:
        tf.config.experimental.set_memory_growth(g, True)
    print(f"✅ GPU(s) détecté(s) : {[g.name for g in gpus]}")
else:
    print("⚠️ Pas de GPU — entraînement sur CPU (plus lent)")

print("TensorFlow :", tf.__version__)
print("OpenCV     :", cv2.__version__)


---
# 2. Configuration du dataset Magnetic Tile

**Structure attendue** (depuis `alex000kim/magnetic-tile-surface-defects`) :

```
.../magnetic-tile-surface-defects/
├── MT_Blowhole/Imgs/
│   ├── exp1_num_xxx.jpg   ← image grayscale
│   └── exp1_num_xxx.png   ← masque binaire (même nom, ext .png)
├── MT_Break/Imgs/
├── MT_Crack/Imgs/
├── MT_Fray/Imgs/
├── MT_Free/Imgs/          ← non défectueux
└── MT_Uneven/Imgs/
```

Le code ci-dessous **auto-détecte** la racine sur Kaggle ou en local.


In [ ]:
# ============================================================
# Auto-détection de la racine du dataset
# ------------------------------------------------------------
# Sur Kaggle, le dataset 'alex000kim/magnetic-tile-surface-defects' est
# attaché à /kaggle/input/magnetic-tile-surface-defects/ . Mais il
# contient un dossier interne 'Magnetic-tile-defect-datasets.' (avec
# le point final !) qui contient les 6 dossiers MT_*. On scanne donc
# récursivement à partir des points d'entrée les plus probables.
# ============================================================
SEARCH_ROOTS = [
    "/kaggle/input",                                          # Kaggle : toujours scanner ici
    "/kaggle/input/magnetic-tile-surface-defects",
    "./Magnetic-tile-defect-datasets",                        # local
    "./Magnetic-tile-defect-datasets.",                       # local (avec point)
    "./magnetic-tile-surface-defects",                        # local
    "../input/magnetic-tile-surface-defects",                 # fallback
    ".",                                                      # cwd
]

def autodetect_dataset_root(search_roots):
    """Cherche le dossier contenant >=3 sous-dossiers MT_<class>."""
    seen = set()
    for sr in search_roots:
        if not os.path.isdir(sr):
            continue
        for dirpath, dirnames, _ in os.walk(sr):
            if dirpath in seen:
                continue
            seen.add(dirpath)
            mt_dirs = [d for d in dirnames if d.startswith("MT_")]
            if len(mt_dirs) >= 3:
                return dirpath, mt_dirs
    return None, None


DATA_ROOT, mt_found = autodetect_dataset_root(SEARCH_ROOTS)

if DATA_ROOT is None:
    # Diagnostic : afficher ce qu'on voit dans /kaggle/input pour aider
    print("❌ Dataset non trouvé. État de /kaggle/input :")
    try:
        for d in sorted(os.listdir("/kaggle/input")):
            sub = f"/kaggle/input/{d}"
            print(f"  {sub}")
            if os.path.isdir(sub):
                for s in sorted(os.listdir(sub))[:10]:
                    print(f"    └─ {s}")
    except Exception as e:
        print(f"  (impossible de lister /kaggle/input : {e})")
    raise FileNotFoundError(
        "Attachez la dataset 'magnetic-tile-surface-defects' (alex000kim) "
        "dans le panneau 'Add data' à droite, puis re-exécutez cette cellule."
    )

print(f"✅ DATA_ROOT détecté : {DATA_ROOT}")
print(f"   Sous-dossiers MT_* trouvés : {sorted(mt_found)}")
print(f"   Contenu (extrait) : {sorted(os.listdir(DATA_ROOT))[:10]}")

# ------------------------------------------------------------
# Constantes globales
# ------------------------------------------------------------
IMG_SIZE = 224
CHANNELS = 1
BATCH_SIZE = 32
EPOCHS = 50
LR = 1e-3

CLASS_NAMES = ["Blowhole", "Break", "Crack", "Fray", "Free", "Uneven"]
CLASS_COLORS = ["#E74C3C", "#3498DB", "#9B59B6", "#F39C12", "#27AE60", "#1ABC9C"]
LABEL_TO_ID = {n: i for i, n in enumerate(CLASS_NAMES)}
ID_TO_LABEL = {i: n for n, i in LABEL_TO_ID.items()}
NUM_CLASSES = len(CLASS_NAMES)

# Mapping des dossiers vers labels canoniques
FOLDER_TO_LABEL = {
    "MT_Blowhole": "Blowhole",
    "MT_Break":    "Break",
    "MT_Crack":    "Crack",
    "MT_Fray":     "Fray",
    "MT_Free":     "Free",
    "MT_Uneven":   "Uneven",
}

print(f"\n📋 {NUM_CLASSES} classes : {CLASS_NAMES}")


---
# 3. Chargement récursif des images + masques

Particularité Magnetic Tile : dans chaque `MT_<class>/Imgs/` se trouvent **deux fichiers par échantillon** :
- `name.jpg` → image grayscale
- `name.png` → masque binaire de la zone défectueuse (sauf pour `MT_Free`)

On charge **les deux** : l'image pour la classification, le masque pour la validation Grad-CAM plus tard.


In [ ]:
def find_pair_files(root, folder_to_label):
    """Retourne (img_path, mask_path_or_None, label) pour chaque sample.

    Cherche les dossiers MT_<class>/Imgs/ ou MT_<class>/ directement.
    Convention Magnetic Tile : chaque image .jpg a un masque .png homonyme
    (sauf MT_Free qui n'a pas de défaut donc pas toujours de masque).
    """
    records = []
    for folder, label in folder_to_label.items():
        folder_path = os.path.join(root, folder)
        if not os.path.isdir(folder_path):
            print(f"⚠️ Dossier introuvable : {folder_path}")
            continue

        # Préférence pour MT_<class>/Imgs/, sinon MT_<class>/ directement
        candidate = os.path.join(folder_path, "Imgs")
        imgs_dir = candidate if os.path.isdir(candidate) else folder_path

        # Walk au cas où Imgs/ contient encore d'autres niveaux
        all_files = []
        for dp, _, fs in os.walk(imgs_dir):
            for f in fs:
                all_files.append(os.path.join(dp, f))

        jpgs = sorted(f for f in all_files if f.lower().endswith((".jpg", ".jpeg", ".bmp")))
        pngs = set(f for f in all_files if f.lower().endswith(".png"))

        # On indexe les pngs par leur stem pour matcher rapidement
        png_by_stem = {os.path.splitext(os.path.basename(p))[0]: p for p in pngs}

        for jpg in jpgs:
            stem = os.path.splitext(os.path.basename(jpg))[0]
            mask_path = png_by_stem.get(stem)
            records.append({
                "img_path": jpg,
                "mask_path": mask_path,
                "label": label,
                "label_id": LABEL_TO_ID[label],
                "stem": stem,
            })
        print(f"  {folder:<14} -> {len([r for r in records if r['label']==label])} images")
    return records


records = find_pair_files(DATA_ROOT, FOLDER_TO_LABEL)
df_meta = pd.DataFrame(records)
print(f"✅ {len(df_meta)} échantillons trouvés")
print(df_meta["label"].value_counts())
print(f"\nAvec masque : {df_meta['mask_path'].notna().sum()} / {len(df_meta)}")


In [ ]:
def load_image_gray(path, img_size=224):
    img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return None
    img = cv2.resize(img, (img_size, img_size), interpolation=cv2.INTER_AREA)
    return img.astype(np.float32) / 255.0


def load_mask_binary(path, img_size=224):
    if path is None:
        return np.zeros((img_size, img_size), dtype=np.uint8)
    m = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if m is None:
        return np.zeros((img_size, img_size), dtype=np.uint8)
    m = cv2.resize(m, (img_size, img_size), interpolation=cv2.INTER_NEAREST)
    return (m > 127).astype(np.uint8)


print("⏳ Chargement des images et masques en RAM...")
X_list, M_list, y_list, stems = [], [], [], []
orig_sizes = []

for r in tqdm(records, desc="Loading"):
    raw = cv2.imread(r["img_path"], cv2.IMREAD_GRAYSCALE)
    if raw is None:
        continue
    orig_sizes.append(raw.shape)
    img = cv2.resize(raw, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA).astype(np.float32) / 255.0
    mask = load_mask_binary(r["mask_path"], IMG_SIZE)
    X_list.append(img[..., np.newaxis])
    M_list.append(mask)
    y_list.append(r["label_id"])
    stems.append(r["stem"])

X_all = np.stack(X_list).astype(np.float32)
M_all = np.stack(M_list).astype(np.uint8)
y_all = np.array(y_list, dtype=np.int32)

print(f"\nShape X_all : {X_all.shape}")
print(f"Shape M_all : {M_all.shape}  (binary masks)")
print(f"Shape y_all : {y_all.shape}")
print(f"Plage X     : [{X_all.min():.3f}, {X_all.max():.3f}]")
print(f"Distribution : {dict(Counter(y_all))}")


---
# 4. Analyse exploratoire (EDA)


In [ ]:
# Distribution des classes
counts = Counter(y_all)
df_dist = pd.DataFrame({
    "Classe": CLASS_NAMES,
    "Nombre": [counts.get(i, 0) for i in range(NUM_CLASSES)],
})
df_dist["Pourcentage"] = (df_dist["Nombre"] / df_dist["Nombre"].sum() * 100).round(1)
print(df_dist.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

bars = axes[0].bar(CLASS_NAMES, df_dist["Nombre"], color=CLASS_COLORS, edgecolor="black")
axes[0].set_title("Distribution des classes (DÉSÉQUILIBRÉE)", fontsize=12, fontweight="bold")
axes[0].set_ylabel("Nombre d'images")
axes[0].tick_params(axis="x", rotation=30)
for b, v in zip(bars, df_dist["Nombre"]):
    axes[0].text(b.get_x() + b.get_width()/2, b.get_height(), str(v),
                 ha="center", va="bottom", fontsize=10)

axes[1].pie(df_dist["Nombre"], labels=CLASS_NAMES, colors=CLASS_COLORS,
            autopct="%1.1f%%", startangle=90)
axes[1].set_title("Répartition proportionnelle", fontsize=12, fontweight="bold")

plt.suptitle("📊 Distribution Magnetic Tile Defect", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

ratio = df_dist["Nombre"].max() / df_dist["Nombre"].min()
print(f"\n📌 Ratio max/min = {ratio:.1f}x")
print("⚠️ Déséquilibre SÉVÈRE — nécessite class_weight et/ou Focal Loss.")


In [ ]:
# Distribution des tailles d'images originales (avant resize)
sizes_df = pd.DataFrame(orig_sizes, columns=["H", "W"])
sizes_df["aspect"] = sizes_df["W"] / sizes_df["H"]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(sizes_df["H"], bins=30, color="#3498DB", edgecolor="black")
axes[0].set_title("Hauteur originale (px)")
axes[0].set_xlabel("H"); axes[0].set_ylabel("Count")

axes[1].hist(sizes_df["W"], bins=30, color="#E74C3C", edgecolor="black")
axes[1].set_title("Largeur originale (px)")
axes[1].set_xlabel("W")

axes[2].hist(sizes_df["aspect"], bins=30, color="#27AE60", edgecolor="black")
axes[2].set_title("Aspect ratio (W/H)")
axes[2].set_xlabel("W/H")

plt.suptitle("Variabilité des dimensions originales", fontweight="bold")
plt.tight_layout()
plt.show()

print(f"Hauteur  : min={sizes_df['H'].min()}, max={sizes_df['H'].max()}, median={sizes_df['H'].median()}")
print(f"Largeur  : min={sizes_df['W'].min()}, max={sizes_df['W'].max()}, median={sizes_df['W'].median()}")
print(f"\n📌 Le resize uniforme {IMG_SIZE}x{IMG_SIZE} est donc indispensable.")


In [ ]:
# Échantillons par classe + leurs masques GT
N_SAMPLES = 3
fig, axes = plt.subplots(NUM_CLASSES, N_SAMPLES * 2, figsize=(N_SAMPLES * 5, NUM_CLASSES * 2.2))

for cls in range(NUM_CLASSES):
    idxs = np.where(y_all == cls)[0]
    chosen = np.random.choice(idxs, min(N_SAMPLES, len(idxs)), replace=False)
    for j, idx in enumerate(chosen):
        axes[cls, 2*j].imshow(X_all[idx].squeeze(), cmap="gray")
        axes[cls, 2*j].set_title(f"{CLASS_NAMES[cls]}", fontsize=9,
                                  color=CLASS_COLORS[cls], fontweight="bold")
        axes[cls, 2*j].axis("off")

        # Overlay du masque
        overlay = cv2.cvtColor((X_all[idx].squeeze() * 255).astype(np.uint8),
                               cv2.COLOR_GRAY2RGB)
        m = M_all[idx]
        overlay[m > 0] = [255, 0, 0]
        axes[cls, 2*j + 1].imshow(overlay)
        axes[cls, 2*j + 1].set_title("+ masque GT", fontsize=9)
        axes[cls, 2*j + 1].axis("off")

plt.suptitle("🔍 Échantillons et masques de défauts par classe", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("eda_samples.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 5. Prétraitement CLAHE

Les images magnetic tile ont un contraste très variable selon l'éclairage industriel.
**CLAHE** (*Contrast Limited Adaptive Histogram Equalization*) améliore la lisibilité locale
sans amplifier le bruit.


In [ ]:
def apply_clahe_batch(X, clip_limit=2.5, tile_grid_size=(8, 8)):
    clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=tile_grid_size)
    out = np.empty_like(X)
    for i, img in enumerate(X):
        u8 = (img.squeeze() * 255).astype(np.uint8)
        out[i] = (clahe.apply(u8) / 255.0)[..., np.newaxis]
    return out


print("⏳ Application CLAHE...")
X_all_c = apply_clahe_batch(X_all, clip_limit=2.5)
print(f"✅ Shape après CLAHE : {X_all_c.shape}")

# Comparaison visuelle avant / après
fig, axes = plt.subplots(2, NUM_CLASSES, figsize=(NUM_CLASSES * 2.6, 5))
for cls in range(NUM_CLASSES):
    idx = np.where(y_all == cls)[0][0]
    axes[0, cls].imshow(X_all[idx].squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[0, cls].set_title(CLASS_NAMES[cls], fontsize=10)
    axes[0, cls].axis("off")
    axes[1, cls].imshow(X_all_c[idx].squeeze(), cmap="gray", vmin=0, vmax=1)
    axes[1, cls].axis("off")
axes[0, 0].set_ylabel("Original", fontsize=11)
axes[1, 0].set_ylabel("CLAHE",    fontsize=11)
plt.suptitle("Effet de CLAHE sur chaque classe", fontweight="bold")
plt.tight_layout()
plt.savefig("eda_clahe.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 6. Split stratifié 70/15/15 + gestion du déséquilibre

Avec une classe `Free` qui domine (~71% des images), le split stratifié est **indispensable**.
On calcule également :
- les **class weights** (Sklearn `balanced`) pour pondérer la loss
- une distribution équivalente pour chaque split


In [ ]:
indices = np.arange(len(X_all_c))

idx_train, idx_temp = train_test_split(
    indices, test_size=0.30, stratify=y_all, random_state=SEED
)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.50, stratify=y_all[idx_temp], random_state=SEED
)

X_train = X_all_c[idx_train];  y_train = y_all[idx_train];  M_train = M_all[idx_train]
X_val   = X_all_c[idx_val];    y_val   = y_all[idx_val];    M_val   = M_all[idx_val]
X_test  = X_all_c[idx_test];   y_test  = y_all[idx_test];   M_test  = M_all[idx_test]

y_train_oh = to_categorical(y_train, NUM_CLASSES).astype(np.float32)
y_val_oh   = to_categorical(y_val,   NUM_CLASSES).astype(np.float32)
y_test_oh  = to_categorical(y_test,  NUM_CLASSES).astype(np.float32)

print("=" * 60)
print(f"Train : {len(X_train):4d}  |  Val : {len(X_val):4d}  |  Test : {len(X_test):4d}")
print("=" * 60)
print(f"{'Classe':<12} {'Train':>6} {'Val':>6} {'Test':>6} {'%global':>8}")
for cls in range(NUM_CLASSES):
    tr = (y_train == cls).sum()
    vl = (y_val   == cls).sum()
    te = (y_test  == cls).sum()
    pct = 100 * (tr + vl + te) / len(y_all)
    print(f"{CLASS_NAMES[cls]:<12} {tr:>6} {vl:>6} {te:>6} {pct:>7.1f}%")

# ------------------------------------------------------------
# Class weights — version DOUCE (sqrt-balanced, pas balanced complet)
# ------------------------------------------------------------
# Le ratio Free:Fray ≈ 30:1. Avec class_weight='balanced' classique
# (w_i = N / (K * n_i)) le poids de Fray devient ~30x celui de Free,
# ce qui combiné à la Focal Loss fait COLLAPSE le modèle vers la
# classe rare. On utilise donc la racine carrée des fréquences
# inverses : correction présente mais nettement moins agressive.
# ------------------------------------------------------------
freqs = np.array([(y_train == i).sum() for i in range(NUM_CLASSES)], dtype=np.float32)
inv_freqs = 1.0 / freqs
sqrt_weights = np.sqrt(inv_freqs)
# Normalisation pour que la moyenne pondérée par les fréquences = 1
sqrt_weights = sqrt_weights * len(y_train) / (sqrt_weights * freqs).sum()
class_weight_dict = {i: float(w) for i, w in enumerate(sqrt_weights)}

print("\nClass weights (sqrt-balanced, version douce) :")
for i, w in class_weight_dict.items():
    print(f"  {CLASS_NAMES[i]:<10} -> {w:.3f}  (n={int(freqs[i])})")

# Pour comparaison : les poids 'balanced' classiques (TROP agressifs)
balanced_full = compute_class_weight("balanced", classes=np.arange(NUM_CLASSES), y=y_train)
print("\n  Comparaison — poids 'balanced' classiques (NON utilisés) :")
for i, w in enumerate(balanced_full):
    print(f"  {CLASS_NAMES[i]:<10} -> {w:.3f}")


---
# 🔬 Test diagnostique #1 — Faut-il vraiment corriger le déséquilibre ?

> **Hypothèse** : si on entraîne un CNN sans aucune correction du déséquilibre,
> le modèle va prendre le raccourci "tout prédire = `Free`" car cela donne
> déjà ~71% d'accuracy (raccourci optimal sans incentive).

On entraîne un mini-CNN naïf pendant **8 epochs**, **sans class_weight**, **sans focal loss**,
juste de la cross-entropy pure. On regarde ce que le modèle apprend réellement.


In [ ]:
# Mini-CNN naïf inline — version contrôle pour mesurer l'effet du déséquilibre
def _build_naive_cnn():
    inp = keras.Input((IMG_SIZE, IMG_SIZE, CHANNELS))
    x = layers.Conv2D(32, 3, padding="same", activation="relu")(inp)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(64, 3, padding="same", activation="relu")(x)
    x = layers.MaxPooling2D(2)(x)
    x = layers.Conv2D(128, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(64, activation="relu")(x)
    out = layers.Dense(NUM_CLASSES, activation="softmax")(x)
    m = keras.Model(inp, out, name="Naive_CNN_NoBalancing")
    m.compile(optimizer=optimizers.Adam(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
    return m

naive_cnn = _build_naive_cnn()

print("🧪 Entraînement du modèle NAÏF (cross-entropy pure, sans class_weight)...")
t0 = time.time()
hist_naive = naive_cnn.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=8, batch_size=BATCH_SIZE,
    verbose=0,
)
print(f"⏱️  Entraînement : {time.time() - t0:.0f}s\n")

# Évaluation
pred_naive = np.argmax(naive_cnn.predict(X_test, verbose=0), axis=1)
acc_naive = accuracy_score(y_test, pred_naive)
f1m_naive = f1_score(y_test, pred_naive, average="macro", zero_division=0)

# Distribution des prédictions vs vrai
pred_dist = Counter(pred_naive); true_dist = Counter(y_test)
print(f"📊 Accuracy naïve : {acc_naive:.3f}  |  F1_macro : {f1m_naive:.3f}")
print(f"\n{'Classe':<12} {'Vrai test':>10} {'Prédictions naïves':>20}")
for cls in range(NUM_CLASSES):
    print(f"{CLASS_NAMES[cls]:<12} {true_dist.get(cls, 0):>10} {pred_dist.get(cls, 0):>20}")

# Visualisation
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].bar(CLASS_NAMES, [true_dist.get(i, 0) for i in range(NUM_CLASSES)],
            color="#3498DB", edgecolor="black", label="Vraie distribution test")
axes[0].set_title("Distribution réelle du test set"); axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(CLASS_NAMES, [pred_dist.get(i, 0) for i in range(NUM_CLASSES)],
            color="#E74C3C", edgecolor="black", label="Prédictions naïves")
axes[1].set_title(f"Prédictions modèle naïf  (acc={acc_naive:.2f}, F1={f1m_naive:.2f})")
axes[1].tick_params(axis="x", rotation=30)

plt.suptitle("🚨 Effondrement du modèle naïf vers la classe majoritaire", fontweight="bold")
plt.tight_layout()
plt.savefig("diagnostic_test1_naive.png", dpi=150, bbox_inches="tight")
plt.show()

# Verdict
collapse_class = max(pred_dist, key=pred_dist.get)
collapse_pct = 100 * pred_dist[collapse_class] / len(pred_naive)
print(f"\n🔍 CONSTAT : le modèle prédit '{CLASS_NAMES[collapse_class]}' dans {collapse_pct:.0f}% des cas.")
print(f"   F1_macro = {f1m_naive:.3f}  ≪  F1_macro idéal ≥ 0.6")
print("   ➡️ DÉCISION : on doit corriger le déséquilibre via class_weight + Focal Loss.\n")

# Cleanup
del naive_cnn


---
# 7. Augmentation industrielle + CutMix

Le flip horizontal **est ici acceptable** (contrairement à NEU-DET pour certaines classes),
car les défauts magnetic tile sont sans orientation forte.

On combine :
- rotation ±15°, shift ±10%, zoom [0.9, 1.1], flip H
- bruit gaussien σ∈[0.01, 0.03]
- bruit salt-and-pepper p=0.02
- **Cutout** (mask_size=24)
- **CutMix** (β=1.0) — upgrade over Mixup : préserve les textures locales


In [ ]:
def add_gaussian_noise(img, sigma=None):
    if sigma is None:
        sigma = np.random.uniform(0.01, 0.03)
    return np.clip(img + np.random.normal(0, sigma, img.shape).astype(np.float32), 0, 1)


def add_salt_pepper(img, amount=0.02):
    out = img.copy()
    h, w, c = out.shape
    n = int(amount * h * w)
    ys = np.random.randint(0, h, n // 2); xs = np.random.randint(0, w, n // 2)
    out[ys, xs, :] = 1.0
    ys = np.random.randint(0, h, n // 2); xs = np.random.randint(0, w, n // 2)
    out[ys, xs, :] = 0.0
    return out


def cutout(img, mask_size=24):
    out = img.copy()
    h, w, c = out.shape
    cy = np.random.randint(mask_size // 2, h - mask_size // 2)
    cx = np.random.randint(mask_size // 2, w - mask_size // 2)
    out[cy - mask_size//2: cy + mask_size//2,
        cx - mask_size//2: cx + mask_size//2, :] = 0.5
    return out


def industrial_preprocess(img):
    """Augmentation DOUCE — version atténuée après diagnostic.

    Le premier essai utilisait p=0.7 / 0.4 / 0.4 et un bruit σ jusqu'à 0.03.
    Combiné à CutMix, ça noyait le signal des classes rares (Fray n=22) et
    les CNN from-scratch collapsaient vers 'Free'. On atténue tout.
    """
    img = img.astype(np.float32)
    if np.random.rand() < 0.30: img = add_gaussian_noise(img, sigma=np.random.uniform(0.005, 0.015))
    if np.random.rand() < 0.15: img = add_salt_pepper(img, amount=0.01)
    if np.random.rand() < 0.20: img = cutout(img, mask_size=16)
    return np.clip(img, 0, 1).astype(np.float32)


# Augmentation géométrique douce, sans preprocessing custom
train_datagen = ImageDataGenerator(
    rotation_range=10,
    width_shift_range=0.06,
    height_shift_range=0.06,
    zoom_range=[0.95, 1.05],
    horizontal_flip=True,
    fill_mode="nearest",
    preprocessing_function=industrial_preprocess,
)
val_datagen = ImageDataGenerator()


# ------------------------------------------------------------
# CutMix generator (upgrade over Mixup — locally preserves textures)
# ------------------------------------------------------------
def cutmix_batch(X_batch, y_batch, alpha=1.0):
    B, H, W, _ = X_batch.shape
    lam = np.random.beta(alpha, alpha)
    perm = np.random.permutation(B)

    cut_w = int(W * np.sqrt(1 - lam))
    cut_h = int(H * np.sqrt(1 - lam))
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = max(cx - cut_w // 2, 0); x2 = min(cx + cut_w // 2, W)
    y1 = max(cy - cut_h // 2, 0); y2 = min(cy + cut_h // 2, H)

    X_out = X_batch.copy()
    X_out[:, y1:y2, x1:x2, :] = X_batch[perm, y1:y2, x1:x2, :]

    lam_adjusted = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    y_out = lam_adjusted * y_batch + (1 - lam_adjusted) * y_batch[perm]
    return X_out.astype(np.float32), y_out.astype(np.float32)


class CutMixGenerator(tf.keras.utils.Sequence):
    def __init__(self, X, y_oh, batch_size=32, augment=True, cutmix_prob=0.5, alpha=1.0, shuffle=True):
        self.X = X; self.y_oh = y_oh
        self.batch_size = batch_size
        self.augment = augment
        self.cutmix_prob = cutmix_prob
        self.alpha = alpha
        self.shuffle = shuffle
        self.indices = np.arange(len(X))
        self.datagen = train_datagen if augment else val_datagen

    def __len__(self):
        return int(np.ceil(len(self.X) / self.batch_size))

    def __getitem__(self, idx):
        b_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        Xb = self.X[b_idx]; yb = self.y_oh[b_idx]
        if self.augment:
            Xb = np.stack([self.datagen.random_transform(x) for x in Xb])
            Xb = np.stack([self.datagen.preprocessing_function(x) if self.datagen.preprocessing_function else x for x in Xb])
            if np.random.rand() < self.cutmix_prob and len(Xb) >= 2:
                Xb, yb = cutmix_batch(Xb, yb, alpha=self.alpha)
        return Xb.astype(np.float32), yb.astype(np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# CutMix DÉSACTIVÉ par défaut pour les CNN from-scratch :
# avec n_Fray=22, le mélange de patches crée des labels mixtes que les classes
# rares n'arrivent pas à apprendre depuis zéro. On garde le code pour démo
# mais on entraîne sans (cutmix_prob=0.0).
gen_train = CutMixGenerator(X_train, y_train_oh, BATCH_SIZE, augment=True,  cutmix_prob=0.0)
gen_val   = CutMixGenerator(X_val,   y_val_oh,   BATCH_SIZE, augment=False, shuffle=False)
print(f"✅ Generators : train={len(gen_train)} batches, val={len(gen_val)} batches")
print("   (CutMix désactivé — augmentation géométrique douce uniquement)")

# Visualisation
Xb, yb = gen_train[0]
fig, axes = plt.subplots(1, 6, figsize=(15, 3))
for i in range(6):
    axes[i].imshow(Xb[i].squeeze(), cmap="gray")
    axes[i].set_title(f"~{CLASS_NAMES[int(np.argmax(yb[i]))]}", fontsize=9)
    axes[i].axis("off")
plt.suptitle("Augmentations + CutMix", fontweight="bold")
plt.tight_layout()
plt.show()


---
# 8. Focal Loss + Label Smoothing

**Focal Loss** (Lin et al., 2017) :  $FL(p_t) = -\alpha_t (1-p_t)^\gamma \log(p_t)$.

Avec γ=2, les exemples bien classés (faciles) sont **down-weighted**, ce qui force le modèle
à se concentrer sur les classes rares (`Crack`, `Fray`) qui sont par défaut sous-représentées.


In [ ]:
class CategoricalFocalLoss(tf.keras.losses.Loss):
    """Focal Loss (Lin et al. 2017) — version douce.

    IMPORTANT : on n'utilise PAS le vecteur alpha en plus de class_weight
    dans .fit(), sinon on cumule deux corrections au déséquilibre et le
    modèle s'effondre (predit uniquement les classes rares). Une seule
    correction à la fois.
    """
    def __init__(self, gamma=1.0, label_smoothing=0.05, name="focal_loss"):
        super().__init__(name=name)
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def call(self, y_true, y_pred):
        eps = 1e-7
        y_pred = tf.clip_by_value(y_pred, eps, 1.0 - eps)
        if self.label_smoothing > 0:
            K = tf.cast(tf.shape(y_true)[-1], tf.float32)
            y_true = y_true * (1.0 - self.label_smoothing) + self.label_smoothing / K
        ce = -y_true * tf.math.log(y_pred)
        modulator = tf.pow(1.0 - y_pred, self.gamma)  # gamma=1 (doux), pas 2
        return tf.reduce_sum(modulator * ce, axis=-1)


print("Focal Loss : gamma=1.0, label_smoothing=0.05, SANS alpha")
print("(le déséquilibre est géré uniquement via class_weight dans .fit())")


---
# 9. Baseline — CNN custom from scratch (4 blocs)

Architecture de référence identique à celle de NEU-DET : 4 blocs Conv-BN-ReLU-Pool-Dropout,
filtres `32 → 64 → 128 → 256`, head `GAP → Dense(128) → Dropout → Softmax(6)`.


In [ ]:
def conv_block(x, filters, name):
    x = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_conv")(x)
    x = layers.BatchNormalization(name=f"{name}_bn")(x)
    x = layers.Activation("relu", name=f"{name}_relu")(x)
    x = layers.MaxPooling2D(2, name=f"{name}_pool")(x)
    x = layers.Dropout(0.3, name=f"{name}_drop")(x)
    return x


def build_baseline_cnn(input_shape=(224, 224, 1), num_classes=6, lr=1e-3):
    inp = keras.Input(input_shape, name="input_image")
    x = conv_block(inp, 32,  "block1")
    x = conv_block(x,   64,  "block2")
    x = conv_block(x,   128, "block3")
    x = conv_block(x,   256, "block4")
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(128, activation="relu", name="dense_128")(x)
    x = layers.Dropout(0.5, name="head_drop")(x)
    out = layers.Dense(num_classes, activation="softmax", name="predictions")(x)
    m = keras.Model(inp, out, name="Baseline_CNN")
    m.compile(
        optimizer=optimizers.Adam(lr),
        loss="categorical_crossentropy",
        metrics=["accuracy"],
    )
    return m


baseline_cnn = build_baseline_cnn((IMG_SIZE, IMG_SIZE, CHANNELS), NUM_CLASSES, LR)
baseline_cnn.summary()


In [ ]:
baseline_callbacks = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=10,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5,
                                patience=5, min_lr=1e-7, verbose=1),
    callbacks.ModelCheckpoint("best_baseline_cnn.keras",
                              monitor="val_accuracy", save_best_only=True, verbose=0),
]

print("🚀 Entraînement Baseline CNN (direct, sans générateur)...")
print("   Note : on entraîne directement sur X_train pour éviter le décalage")
print("   train/val que créait le générateur d'augmentation sur les classes rares.")
t0 = time.time()
history_baseline = baseline_cnn.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=baseline_callbacks,
    verbose=1,
)
print(f"\n⏱️ Temps total : {time.time() - t0:.0f}s")


---
# 10. CNN amélioré — Résiduel + SE + **CBAM**

**Upgrade vs NEU-DET** : ajout d'un bloc **CBAM** (Convolutional Block Attention Module)
qui combine attention **canal** ET attention **spatiale**, alors que SE seul ne fait
que de l'attention canal.

> CBAM = SE → Spatial Attention(max+avg pool over channels → conv 7×7 → sigmoid)


In [ ]:
def se_block(x, ratio=8):
    c = x.shape[-1]
    s = layers.GlobalAveragePooling2D()(x)
    s = layers.Dense(max(c // ratio, 1), activation="relu")(s)
    s = layers.Dense(c, activation="sigmoid")(s)
    s = layers.Reshape((1, 1, c))(s)
    return layers.Multiply()([x, s])


def spatial_attention(x):
    avg = layers.Lambda(lambda t: tf.reduce_mean(t, axis=-1, keepdims=True))(x)
    mx  = layers.Lambda(lambda t: tf.reduce_max(t, axis=-1, keepdims=True))(x)
    cat = layers.Concatenate(axis=-1)([avg, mx])
    a = layers.Conv2D(1, 7, padding="same", activation="sigmoid")(cat)
    return layers.Multiply()([x, a])


def cbam_block(x, ratio=8):
    x = se_block(x, ratio=ratio)
    x = spatial_attention(x)
    return x


def residual_cbam_block(x, filters, name):
    shortcut = x
    out = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_c1")(x)
    out = layers.BatchNormalization(name=f"{name}_bn1")(out)
    out = layers.Activation("relu", name=f"{name}_r1")(out)
    out = layers.Conv2D(filters, 3, padding="same", use_bias=False, name=f"{name}_c2")(out)
    out = layers.BatchNormalization(name=f"{name}_bn2")(out)
    out = cbam_block(out, ratio=8)
    if shortcut.shape[-1] != filters:
        shortcut = layers.Conv2D(filters, 1, padding="same", use_bias=False,
                                 name=f"{name}_proj")(shortcut)
        shortcut = layers.BatchNormalization(name=f"{name}_proj_bn")(shortcut)
    out = layers.Add(name=f"{name}_add")([out, shortcut])
    out = layers.Activation("relu", name=f"{name}_out")(out)
    out = layers.MaxPooling2D(2, name=f"{name}_pool")(out)
    out = layers.Dropout(0.10, name=f"{name}_drop")(out)   # 0.25 → 0.10 (anti-collapse)
    return out


def build_improved_cnn(input_shape=(224, 224, 1), num_classes=6, lr=5e-4):
    inp = keras.Input(input_shape, name="input_image")
    x = layers.Conv2D(32, 7, strides=2, padding="same", use_bias=False, name="stem_conv")(inp)
    x = layers.BatchNormalization(name="stem_bn")(x)
    x = layers.Activation("relu", name="stem_relu")(x)
    x = layers.MaxPooling2D(3, strides=2, padding="same", name="stem_pool")(x)

    x = residual_cbam_block(x, 64,  "block1")
    x = residual_cbam_block(x, 128, "block2")
    x = residual_cbam_block(x, 256, "block3")   # last conv = Grad-CAM target
    x = residual_cbam_block(x, 512, "block4")

    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dense(256, name="dense_256")(x)
    x = layers.BatchNormalization(name="dense_bn")(x)
    x = layers.Activation("relu", name="dense_relu")(x)
    x = layers.Dropout(0.3, name="dense_drop")(x)   # 0.4 → 0.3
    out = layers.Dense(num_classes, activation="softmax", name="predictions")(x)

    m = keras.Model(inp, out, name="Improved_CNN_ResidualCBAM")
    m.compile(
        optimizer=optimizers.Adam(lr),
        loss=CategoricalFocalLoss(gamma=1.0, label_smoothing=0.05),
        metrics=["accuracy"],
    )
    return m


improved_cnn = build_improved_cnn((IMG_SIZE, IMG_SIZE, CHANNELS), NUM_CLASSES, lr=5e-4)
print(f"Paramètres totaux : {improved_cnn.count_params():,}")


---
# 11. Entraînement du CNN amélioré — Cosine LR avec warmup


In [ ]:
class CosineLRWithWarmup(tf.keras.callbacks.Callback):
    def __init__(self, total_epochs, warmup_epochs=3, lr_max=5e-4, lr_min=1e-7):
        super().__init__()
        self.total_epochs = total_epochs
        self.warmup_epochs = warmup_epochs
        self.lr_max = lr_max
        self.lr_min = lr_min

    def on_epoch_begin(self, epoch, logs=None):
        if epoch < self.warmup_epochs:
            lr = self.lr_max * (epoch + 1) / self.warmup_epochs
        else:
            progress = (epoch - self.warmup_epochs) / max(1, self.total_epochs - self.warmup_epochs)
            lr = self.lr_min + 0.5 * (self.lr_max - self.lr_min) * (1 + math.cos(math.pi * progress))
        try:
            tf.keras.backend.set_value(self.model.optimizer.learning_rate, lr)
        except Exception:
            self.model.optimizer.learning_rate = lr
        if epoch % 5 == 0:
            print(f"   📉 epoch {epoch:2d}  lr = {lr:.2e}")


improved_callbacks = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=12,
                            restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint("best_improved_cnn.keras",
                              monitor="val_accuracy", save_best_only=True, verbose=0),
    CosineLRWithWarmup(total_epochs=EPOCHS, warmup_epochs=3, lr_max=5e-4, lr_min=1e-7),
]

print("🚀 Entraînement Improved CNN (Focal Loss + CBAM + Cosine LR, direct)...")
t0 = time.time()
history_improved = improved_cnn.fit(
    X_train, y_train_oh,
    validation_data=(X_val, y_val_oh),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=improved_callbacks,
    verbose=1,
)
print(f"\n⏱️ Temps total : {time.time() - t0:.0f}s")


In [ ]:
# Courbes d'apprentissage des 2 CNN
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

for i, (hist, name) in enumerate([
    (history_baseline, "Baseline CNN"),
    (history_improved, "Improved CNN (Residual+CBAM, Focal Loss)"),
]):
    df_h = pd.DataFrame(hist.history)
    axes[i, 0].plot(df_h["accuracy"], label="Train")
    axes[i, 0].plot(df_h["val_accuracy"], label="Val")
    axes[i, 0].set_title(f"{name} — Accuracy"); axes[i, 0].set_xlabel("Epoch")
    axes[i, 0].legend(); axes[i, 0].grid(True)
    axes[i, 1].plot(df_h["loss"], label="Train")
    axes[i, 1].plot(df_h["val_loss"], label="Val")
    axes[i, 1].set_title(f"{name} — Loss"); axes[i, 1].set_xlabel("Epoch")
    axes[i, 1].legend(); axes[i, 1].grid(True)

plt.tight_layout()
plt.savefig("learning_curves_cnns.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 12. Transfer Learning — EfficientNetV2B0 (2 stages)

Conversion grayscale → RGB par réplication (toujours après CLAHE),
puis fine-tuning en 2 étapes : head puis 30 dernières couches du backbone.


In [ ]:
from tensorflow.keras.applications.efficientnet_v2 import preprocess_input as eff_preprocess
from tensorflow.keras.applications import EfficientNetV2B0


def to_rgb_preprocessed(X_gray):
    u8 = (X_gray.squeeze(-1) * 255).astype(np.uint8)
    rgb = np.repeat(u8[..., None], 3, axis=-1).astype(np.float32)
    return eff_preprocess(rgb)


X_train_rgb = to_rgb_preprocessed(X_train)
X_val_rgb   = to_rgb_preprocessed(X_val)
X_test_rgb  = to_rgb_preprocessed(X_test)
print(f"✅ RGB shapes : {X_train_rgb.shape}, {X_val_rgb.shape}, {X_test_rgb.shape}")


def build_transfer_model(input_shape=(224, 224, 3), num_classes=6, lr=1e-3):
    base = EfficientNetV2B0(include_top=False, weights="imagenet",
                            input_shape=input_shape, pooling=None)
    base.trainable = False
    inp = keras.Input(input_shape, name="input_rgb")
    x = base(inp, training=False)
    x = layers.GlobalAveragePooling2D(name="gap_tl")(x)
    x = layers.BatchNormalization(name="bn_tl")(x)
    x = layers.Dropout(0.3, name="drop1_tl")(x)
    x = layers.Dense(256, activation="relu", name="dense_tl")(x)
    x = layers.Dropout(0.4, name="drop2_tl")(x)
    out = layers.Dense(num_classes, activation="softmax", name="predictions_tl")(x)
    m = keras.Model(inp, out, name="EfficientNetV2B0_Transfer")
    m.compile(
        optimizer=optimizers.Adam(lr),
        loss=CategoricalFocalLoss(gamma=1.0, label_smoothing=0.05),
        metrics=["accuracy"],
    )
    return m, base


tl_model, base_model = build_transfer_model((IMG_SIZE, IMG_SIZE, 3), NUM_CLASSES, lr=1e-3)
print(f"📦 {tl_model.name}  |  Paramètres : {tl_model.count_params():,}")

stage1_cb = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=6,
                            restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint("best_efficientnet_head.keras",
                              monitor="val_accuracy", save_best_only=True, verbose=0),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3,
                                min_lr=1e-6, verbose=1),
]

print("\n🚀 Stage 1 — head only (backbone frozen)...")
t0 = time.time()
history_head = tl_model.fit(
    X_train_rgb, y_train_oh,
    validation_data=(X_val_rgb, y_val_oh),
    epochs=15, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=stage1_cb, verbose=1,
)
print(f"⏱️ Stage 1 : {time.time() - t0:.0f}s")


In [ ]:
# Stage 2 — fine-tuning des 30 dernières couches
tl_model.load_weights("best_efficientnet_head.keras")

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False
for layer in base_model.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

tl_model.compile(
    optimizer=optimizers.Adam(1e-5),
    loss=CategoricalFocalLoss(gamma=1.0, label_smoothing=0.05),
    metrics=["accuracy"],
)

stage2_cb = [
    callbacks.EarlyStopping(monitor="val_accuracy", patience=8,
                            restore_best_weights=True, verbose=1),
    callbacks.ModelCheckpoint("best_efficientnet_ft.keras",
                              monitor="val_accuracy", save_best_only=True, verbose=0),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=4,
                                min_lr=1e-7, verbose=1),
]

print("🚀 Stage 2 — fine-tuning (LR=1e-5)...")
t0 = time.time()
history_ft = tl_model.fit(
    X_train_rgb, y_train_oh,
    validation_data=(X_val_rgb, y_val_oh),
    epochs=30, batch_size=BATCH_SIZE,
    class_weight=class_weight_dict,
    callbacks=stage2_cb, verbose=1,
)
print(f"⏱️ Stage 2 : {time.time() - t0:.0f}s")

# Courbes combinées
hist_h = pd.DataFrame(history_head.history)
hist_f = pd.DataFrame(history_ft.history)
n_h = len(hist_h)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(range(n_h), hist_h["accuracy"], label="Train S1", color="#3498DB")
axes[0].plot(range(n_h), hist_h["val_accuracy"], label="Val S1", color="#2980B9", linestyle="--")
axes[0].plot(range(n_h, n_h + len(hist_f)), hist_f["accuracy"], label="Train S2", color="#E74C3C")
axes[0].plot(range(n_h, n_h + len(hist_f)), hist_f["val_accuracy"], label="Val S2", color="#C0392B", linestyle="--")
axes[0].axvline(n_h - 0.5, color="gray", linestyle=":", alpha=0.7)
axes[0].set_title("EfficientNetV2 — Accuracy"); axes[0].legend(); axes[0].grid(True)

axes[1].plot(range(n_h), hist_h["loss"], label="Train S1", color="#3498DB")
axes[1].plot(range(n_h), hist_h["val_loss"], label="Val S1", color="#2980B9", linestyle="--")
axes[1].plot(range(n_h, n_h + len(hist_f)), hist_f["loss"], label="Train S2", color="#E74C3C")
axes[1].plot(range(n_h, n_h + len(hist_f)), hist_f["val_loss"], label="Val S2", color="#C0392B", linestyle="--")
axes[1].axvline(n_h - 0.5, color="gray", linestyle=":", alpha=0.7)
axes[1].set_title("EfficientNetV2 — Loss"); axes[1].legend(); axes[1].grid(True)

plt.tight_layout()
plt.savefig("learning_curves_transfer.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 13. Test-Time Augmentation + Évaluation finale


In [ ]:
def default_tta(img):
    img = img.copy()
    if np.random.rand() > 0.5: img = np.fliplr(img)
    if np.random.rand() > 0.5: img = np.flipud(img)
    noise = np.random.normal(0, 0.01, img.shape).astype(np.float32)
    return np.clip(img + noise, 0, 1)


def tta_predict_gray(model, X, n=10):
    probs = np.zeros((len(X), NUM_CLASSES), dtype=np.float32)
    probs += model.predict(X, verbose=0)
    for _ in range(n - 1):
        Xa = np.stack([default_tta(img) for img in X])
        probs += model.predict(Xa, verbose=0)
    return probs / n


def tta_predict_rgb(model, X_rgb, X_gray, n=10):
    probs = model.predict(X_rgb, verbose=0).astype(np.float32)
    for _ in range(n - 1):
        Xa_gray = np.stack([default_tta(img) for img in X_gray])
        Xa_rgb = to_rgb_preprocessed(Xa_gray)
        probs += model.predict(Xa_rgb, verbose=0)
    return probs / n


# Reload best
baseline_cnn.load_weights("best_baseline_cnn.keras")
improved_cnn.load_weights("best_improved_cnn.keras")
tl_model.load_weights("best_efficientnet_ft.keras")

print("📊 Prédictions...")
pred_baseline = np.argmax(baseline_cnn.predict(X_test, verbose=0), axis=1)
pred_improved = np.argmax(improved_cnn.predict(X_test, verbose=0), axis=1)
pred_improved_tta = np.argmax(tta_predict_gray(improved_cnn, X_test, n=10), axis=1)
proba_tl_simple = tl_model.predict(X_test_rgb, verbose=0)
pred_tl_simple = np.argmax(proba_tl_simple, axis=1)
proba_tl_tta = tta_predict_rgb(tl_model, X_test_rgb, X_test, n=10)
pred_tl_tta = np.argmax(proba_tl_tta, axis=1)

results = {
    "Baseline CNN":                  pred_baseline,
    "Improved CNN (CBAM+Focal)":     pred_improved,
    "Improved CNN + TTA":            pred_improved_tta,
    "EfficientNetV2B0":              pred_tl_simple,
    "EfficientNetV2B0 + TTA":        pred_tl_tta,
}

rows = []
for name, pred in results.items():
    acc = accuracy_score(y_test, pred)
    f1_m = f1_score(y_test, pred, average="macro", zero_division=0)
    f1_w = f1_score(y_test, pred, average="weighted", zero_division=0)
    rows.append([name, acc, f1_m, f1_w])

df_results = pd.DataFrame(rows, columns=["Modèle", "Accuracy", "F1_macro", "F1_weighted"])
print("\n" + "=" * 75)
print(df_results.to_string(index=False, float_format="%.4f"))
print("=" * 75)

best_idx = df_results["F1_macro"].idxmax()
best_name = df_results.loc[best_idx, "Modèle"]
print(f"\n🏆 Meilleur modèle (F1 macro — important en cas de déséquilibre) : {best_name}")


In [ ]:
# Rapport détaillé + matrice de confusion pour le meilleur modèle
best_pred = results[best_name]

print(f"📋 Rapport détaillé — {best_name}\n")
print(classification_report(y_test, best_pred, target_names=CLASS_NAMES, zero_division=0))

cm = confusion_matrix(y_test, best_pred, labels=list(range(NUM_CLASSES)))
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True).clip(min=1)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[0])
axes[0].set_title("Matrice de confusion (counts)"); axes[0].set_xlabel("Prédit"); axes[0].set_ylabel("Réel")

sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Reds",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=axes[1])
axes[1].set_title("Matrice de confusion (normalisée par ligne)")
axes[1].set_xlabel("Prédit"); axes[1].set_ylabel("Réel")

plt.suptitle(f"Confusion — {best_name}", fontweight="bold")
plt.tight_layout()
plt.savefig("confusion_best.png", dpi=150, bbox_inches="tight")
plt.show()

# Per-class breakdown
prec, rec, f1, supp = precision_recall_fscore_support(y_test, best_pred, zero_division=0)
df_per_class = pd.DataFrame({
    "Classe": CLASS_NAMES,
    "Support": supp,
    "Precision": prec.round(3),
    "Recall": rec.round(3),
    "F1": f1.round(3),
})
print("\n", df_per_class.to_string(index=False))


---
# 🔬 Test diagnostique #2 — Où le meilleur modèle se trompe-t-il ?

> **Question** : sur quelles paires de classes se produisent les confusions ?
> Les erreurs sont-elles concentrées (= problème ciblé) ou diffuses (= sous-capacité du modèle) ?

On choisit le meilleur modèle, on inspecte les **8 erreurs les plus confiantes**
(le modèle s'est trompé tout en étant sûr de lui) et on calcule la **paire de classes la plus confondue**.


In [ ]:
# Probas du meilleur modèle (TL+TTA si dispo, sinon TL simple, sinon Improved CNN)
best_probs = proba_tl_tta if "proba_tl_tta" in dir() else (
    proba_tl_simple if "proba_tl_simple" in dir() else improved_cnn.predict(X_test, verbose=0)
)
best_pred = np.argmax(best_probs, axis=1)
best_conf = best_probs.max(axis=1)

# ------------------------------------------------------------
# A) Identifier la paire la plus confondue
# ------------------------------------------------------------
cm_err = confusion_matrix(y_test, best_pred, labels=list(range(NUM_CLASSES))).astype(int)
np.fill_diagonal(cm_err, 0)
worst_pairs = []
for i in range(NUM_CLASSES):
    for j in range(NUM_CLASSES):
        if i != j and cm_err[i, j] > 0:
            worst_pairs.append((CLASS_NAMES[i], CLASS_NAMES[j], int(cm_err[i, j])))
worst_pairs.sort(key=lambda t: -t[2])

print("📊 Top-5 paires de classes confondues :")
print(f"{'Vrai':<12} → {'Prédit':<12} {'Erreurs':>8}")
for v, p, n in worst_pairs[:5]:
    print(f"{v:<12} → {p:<12} {n:>8}")

if worst_pairs:
    top_pair = worst_pairs[0]
    print(f"\n🔍 CONSTAT : {top_pair[2]} confusions sur la paire {top_pair[0]} ↔ {top_pair[1]}")
else:
    print("\n🎯 Aucune erreur sur le test set (cas idéal — vérifier le split)")

# ------------------------------------------------------------
# B) Distribution des confidences : correct vs faux
# ------------------------------------------------------------
correct_mask = (best_pred == y_test)
conf_correct = best_conf[correct_mask]
conf_wrong = best_conf[~correct_mask]

print(f"\nConfiance moyenne | correctes : {conf_correct.mean():.3f}  | erronées : {conf_wrong.mean():.3f}")
print(f"Confiance médiane | correctes : {np.median(conf_correct):.3f}  | erronées : {np.median(conf_wrong):.3f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].hist(conf_correct, bins=20, alpha=0.7, color="#27AE60", edgecolor="black", label="Correct")
axes[0].hist(conf_wrong,   bins=20, alpha=0.7, color="#E74C3C", edgecolor="black", label="Erreur")
axes[0].set_xlabel("Confiance prédite"); axes[0].set_ylabel("Fréquence")
axes[0].set_title("Histogramme des confidences"); axes[0].legend(); axes[0].grid(alpha=0.4)

axes[1].boxplot([conf_correct, conf_wrong], labels=["Correct", "Erreur"], patch_artist=True,
                boxprops=dict(facecolor="#3498DB", alpha=0.7))
axes[1].set_ylabel("Confiance"); axes[1].set_title("Distribution boxplot")
axes[1].grid(alpha=0.4)

plt.suptitle("Confiance du modèle quand il a tort vs raison", fontweight="bold")
plt.tight_layout(); plt.savefig("diagnostic_test2_confidence.png", dpi=150, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# C) Les 8 erreurs les plus confiantes (= les plus graves)
# ------------------------------------------------------------
wrong_idx = np.where(~correct_mask)[0]
if len(wrong_idx) > 0:
    # Tri par confiance décroissante
    wrong_sorted = wrong_idx[np.argsort(-best_conf[wrong_idx])]
    top_wrong = wrong_sorted[:min(8, len(wrong_sorted))]
    n_show = len(top_wrong)
    cols = 4; rows = int(np.ceil(n_show / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 3.2, rows * 3.4))
    axes = axes.flatten() if rows > 1 else [axes] if cols == 1 else axes
    for k, idx in enumerate(top_wrong):
        ax = axes[k]
        ax.imshow(X_test[idx].squeeze(), cmap="gray")
        true_name = CLASS_NAMES[y_test[idx]]
        pred_name = CLASS_NAMES[best_pred[idx]]
        ax.set_title(f"Vrai: {true_name}\nPrédit: {pred_name}\nconf={best_conf[idx]:.2f}",
                     fontsize=9, color="#E74C3C")
        ax.axis("off")
    for k in range(n_show, len(axes)):
        axes[k].axis("off")
    plt.suptitle("🚨 Les 8 erreurs les plus confiantes (le modèle s'est trompé en étant sûr)",
                 fontweight="bold")
    plt.tight_layout(); plt.savefig("diagnostic_test2_worst_errors.png", dpi=150, bbox_inches="tight")
    plt.show()

# ------------------------------------------------------------
# VERDICT NARRATIF
# ------------------------------------------------------------
print("\n📋 NARRATIF POUR PRÉSENTATION :")
if worst_pairs:
    print(f"   • La confusion dominante est {worst_pairs[0][0]} → {worst_pairs[0][1]}")
    print(f"     ({worst_pairs[0][2]} échantillons mal classés)")
print(f"   • Le modèle est en moyenne {(conf_correct.mean() - conf_wrong.mean())*100:.1f} pts")
print(f"     plus confiant quand il a raison que quand il se trompe →")
print(f"     un seuil de confiance peut filtrer beaucoup d'erreurs (voir Test #3).")


---
# 14. Calibration des probabilités — Reliability Diagram + ECE

Un modèle bien calibré devrait être **confiant à 90 % et avoir raison ~90 % du temps**.
On calcule l'**Expected Calibration Error (ECE)** et on trace le reliability diagram.


In [ ]:
def expected_calibration_error(probs, y_true, n_bins=10):
    confidences = probs.max(axis=1)
    predictions = probs.argmax(axis=1)
    accuracies = (predictions == y_true).astype(np.float32)
    ece = 0.0
    bins = np.linspace(0, 1, n_bins + 1)
    bin_centers, bin_accs, bin_confs, bin_counts = [], [], [], []
    for i in range(n_bins):
        mask = (confidences > bins[i]) & (confidences <= bins[i + 1])
        cnt = mask.sum()
        if cnt > 0:
            acc = accuracies[mask].mean()
            conf = confidences[mask].mean()
            ece += (cnt / len(probs)) * abs(acc - conf)
            bin_centers.append((bins[i] + bins[i + 1]) / 2)
            bin_accs.append(acc); bin_confs.append(conf); bin_counts.append(cnt)
    return ece, bin_centers, bin_accs, bin_confs, bin_counts


ece_tl_tta, centers, accs, confs, counts = expected_calibration_error(proba_tl_tta, y_test, n_bins=10)
ece_tl_simple, _, _, _, _ = expected_calibration_error(proba_tl_simple, y_test, n_bins=10)

print(f"ECE EfficientNet (sans TTA) : {ece_tl_simple:.4f}")
print(f"ECE EfficientNet + TTA      : {ece_tl_tta:.4f}")

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], "--", color="gray", label="Calibration parfaite")
ax.bar(centers, accs, width=0.08, color="#3498DB", alpha=0.7,
       edgecolor="black", label=f"Accuracy par bin (ECE={ece_tl_tta:.3f})")
ax.plot(centers, confs, "o-", color="#E74C3C", label="Confiance moyenne")
ax.set_xlabel("Confiance prédite"); ax.set_ylabel("Accuracy réelle")
ax.set_title(f"Reliability Diagram — {best_name}"); ax.legend(); ax.grid(True)
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig("calibration.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 15. Grad-CAM + **IoU avec les masques GT** (innovation propre à ce dataset)

Le dataset fournit des masques pixel-précis des défauts. On peut donc **mesurer quantitativement**
si le modèle attend vraiment la bonne zone :

> **Localization-IoU** : on binarise le heatmap Grad-CAM à seuil 0.5,
> puis on calcule l'IoU avec le masque GT. C'est une mesure objective d'interprétabilité
> qu'aucun autre dataset de défauts ne permet aussi facilement.


In [ ]:
def make_gradcam_heatmap(img_array, model, last_conv_name, pred_index=None):
    grad_model = keras.Model(
        inputs=model.inputs,
        outputs=[model.get_layer(last_conv_name).output, model.output],
    )
    with tf.GradientTape() as tape:
        conv_out, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]
    grads = tape.gradient(class_channel, conv_out)
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))
    conv_out = conv_out[0]
    heatmap = conv_out @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)
    heatmap = tf.maximum(heatmap, 0) / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy()


def heatmap_to_mask(heatmap, target_size=224, threshold=0.5):
    h = cv2.resize(heatmap, (target_size, target_size))
    return (h >= threshold).astype(np.uint8)


def iou(mask_a, mask_b):
    inter = np.logical_and(mask_a, mask_b).sum()
    union = np.logical_or(mask_a, mask_b).sum()
    return inter / union if union > 0 else 0.0


# On utilise le CNN amélioré : dernière conv = block3_out (avant le bloc 4)
LAST_CONV_NAME = "block3_out"

# Calcul IoU sur le test set (en ignorant la classe Free qui n'a pas de défaut)
defect_classes = [i for i, n in enumerate(CLASS_NAMES) if n != "Free"]
ious_per_class = defaultdict(list)

print("🔎 Calcul des IoU Grad-CAM vs masques GT...")
print("   Note : on calcule le heatmap pour la VRAIE classe (pas la prédite),")
print("   pour mesurer si le modèle attend la bonne zone même quand il se trompe.")
for idx in tqdm(range(len(X_test))):
    cls = y_test[idx]
    if cls not in defect_classes:
        continue
    if M_test[idx].sum() == 0:
        continue
    img = X_test[idx][None, ...]
    # Use TRUE class index → heatmap reflects evidence for the actual defect
    hm = make_gradcam_heatmap(img, improved_cnn, LAST_CONV_NAME, pred_index=int(cls))
    # Threshold à 0.3 plutôt que 0.5 — les heatmaps peuvent être diffuses
    hm_mask = heatmap_to_mask(hm, IMG_SIZE, threshold=0.3)
    ious_per_class[cls].append(iou(hm_mask, M_test[idx]))

print("\n📌 IoU moyen Grad-CAM ↔ Masque GT par classe :")
print("-" * 50)
for cls in defect_classes:
    if ious_per_class[cls]:
        vals = np.array(ious_per_class[cls])
        print(f"  {CLASS_NAMES[cls]:<10} : mean={vals.mean():.3f}  median={np.median(vals):.3f}  N={len(vals)}")
    else:
        print(f"  {CLASS_NAMES[cls]:<10} : (pas d'échantillon avec masque)")

all_ious = np.concatenate([np.array(v) for v in ious_per_class.values() if v])
print(f"\n  ➡️ IoU global : {all_ious.mean():.3f}  (médiane {np.median(all_ious):.3f})")


In [ ]:
# Visualisation : original | masque GT | Grad-CAM | overlay
fig, axes = plt.subplots(len(defect_classes), 4, figsize=(14, 3.2 * len(defect_classes)))

for row, cls in enumerate(defect_classes):
    cand_idxs = [i for i in range(len(X_test)) if y_test[i] == cls and M_test[i].sum() > 0]
    if not cand_idxs:
        continue
    # Pick the sample with median IoU among those we computed
    pick_idx = cand_idxs[len(cand_idxs) // 2]
    img = X_test[pick_idx]
    mask_gt = M_test[pick_idx]

    hm = make_gradcam_heatmap(img[None, ...], improved_cnn, LAST_CONV_NAME)
    hm_resized = cv2.resize(hm, (IMG_SIZE, IMG_SIZE))
    hm_mask = (hm_resized >= 0.5).astype(np.uint8)
    iou_val = iou(hm_mask, mask_gt)

    axes[row, 0].imshow(img.squeeze(), cmap="gray"); axes[row, 0].axis("off")
    axes[row, 0].set_title(f"{CLASS_NAMES[cls]}", color=CLASS_COLORS[cls], fontweight="bold")

    axes[row, 1].imshow(mask_gt, cmap="gray"); axes[row, 1].axis("off")
    axes[row, 1].set_title("Masque GT")

    axes[row, 2].imshow(hm_resized, cmap="jet"); axes[row, 2].axis("off")
    axes[row, 2].set_title("Grad-CAM heatmap")

    # Overlay GT (red) + GradCAM (green semi-transparent)
    base = cv2.cvtColor((img.squeeze() * 255).astype(np.uint8), cv2.COLOR_GRAY2RGB)
    base[mask_gt > 0] = [255, 0, 0]
    overlay = base.copy()
    g = np.zeros_like(overlay); g[..., 1] = 255
    blend = np.where(hm_mask[..., None] > 0,
                     (0.5 * overlay + 0.5 * g).astype(np.uint8), overlay)
    axes[row, 3].imshow(blend); axes[row, 3].axis("off")
    axes[row, 3].set_title(f"GT (rouge) vs GradCAM (vert) — IoU={iou_val:.2f}")

plt.suptitle("🔬 Validation visuelle Grad-CAM via les masques GT", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("gradcam_iou.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 16. Visualisation des embeddings — t-SNE


In [ ]:
embedding_model = keras.Model(
    inputs=improved_cnn.input,
    outputs=improved_cnn.get_layer("dense_relu").output,
    name="ImprovedCNN_Embeddings",
)

embeddings = embedding_model.predict(X_test, verbose=0)
print(f"Embeddings shape : {embeddings.shape}")

sim = cosine_similarity(embeddings)
intra, inter = [], []
for i in range(len(y_test)):
    for j in range(i + 1, len(y_test)):
        (intra if y_test[i] == y_test[j] else inter).append(sim[i, j])
print(f"Cosine intra-classe : {np.mean(intra):.3f}")
print(f"Cosine inter-classe : {np.mean(inter):.3f}")
print(f"Séparation Δ = {np.mean(intra) - np.mean(inter):.3f}")

# t-SNE
perp = min(30, max(5, len(embeddings) // 10))
emb2d = TSNE(n_components=2, perplexity=perp, random_state=SEED,
             init="pca", learning_rate="auto").fit_transform(embeddings)

plt.figure(figsize=(9, 7))
for cls in range(NUM_CLASSES):
    mask = y_test == cls
    plt.scatter(emb2d[mask, 0], emb2d[mask, 1],
                c=CLASS_COLORS[cls], label=CLASS_NAMES[cls], alpha=0.75, s=40)
plt.title("t-SNE — embeddings du CNN amélioré (test set)", fontsize=13, fontweight="bold")
plt.legend(); plt.grid(True)
plt.tight_layout()
plt.savefig("tsne_embeddings.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 17. Tests de robustesse

On dégrade artificiellement le test set (bruit gaussien, flou, baisse de luminosité)
et on mesure la chute d'accuracy. Mesure essentielle en industrie où l'éclairage varie.


In [ ]:
def corrupt_gaussian(X, sigma):
    return np.clip(X + np.random.normal(0, sigma, X.shape).astype(np.float32), 0, 1)

def corrupt_blur(X, ksize):
    out = X.copy()
    for i in range(len(X)):
        out[i, ..., 0] = cv2.GaussianBlur(X[i, ..., 0], (ksize, ksize), 0)
    return out

def corrupt_brightness(X, gamma):
    return np.clip(X ** gamma, 0, 1)


configs = [
    ("Noise σ=0.05",    lambda X: corrupt_gaussian(X, 0.05)),
    ("Noise σ=0.10",    lambda X: corrupt_gaussian(X, 0.10)),
    ("Blur k=5",        lambda X: corrupt_blur(X, 5)),
    ("Blur k=11",       lambda X: corrupt_blur(X, 11)),
    ("Bright γ=0.5",    lambda X: corrupt_brightness(X, 0.5)),
    ("Bright γ=2.0",    lambda X: corrupt_brightness(X, 2.0)),
]

rob_rows = []
acc_base = accuracy_score(y_test, np.argmax(improved_cnn.predict(X_test, verbose=0), axis=1))
acc_tl   = accuracy_score(y_test, np.argmax(tl_model.predict(X_test_rgb, verbose=0), axis=1))
rob_rows.append(["Clean", acc_base, acc_tl])

for name, fn in configs:
    Xc = fn(X_test)
    a1 = accuracy_score(y_test, np.argmax(improved_cnn.predict(Xc, verbose=0), axis=1))
    Xc_rgb = to_rgb_preprocessed(Xc)
    a2 = accuracy_score(y_test, np.argmax(tl_model.predict(Xc_rgb, verbose=0), axis=1))
    rob_rows.append([name, a1, a2])

df_rob = pd.DataFrame(rob_rows, columns=["Corruption", "Improved CNN", "EfficientNetV2"])
print(df_rob.to_string(index=False, float_format="%.4f"))

fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(df_rob))
w = 0.35
ax.bar(x - w/2, df_rob["Improved CNN"],     w, label="Improved CNN",     color="#3498DB")
ax.bar(x + w/2, df_rob["EfficientNetV2"],   w, label="EfficientNetV2",   color="#E74C3C")
ax.set_xticks(x); ax.set_xticklabels(df_rob["Corruption"], rotation=25)
ax.set_ylabel("Accuracy"); ax.set_title("Robustesse face aux corruptions", fontweight="bold")
ax.legend(); ax.grid(axis="y", alpha=0.4)
ax.set_ylim(0, 1.0)
for i, (a, b) in enumerate(zip(df_rob["Improved CNN"], df_rob["EfficientNetV2"])):
    ax.text(i - w/2, a + 0.01, f"{a:.2f}", ha="center", fontsize=8)
    ax.text(i + w/2, b + 0.01, f"{b:.2f}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("robustness.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 🔬 Test diagnostique #3 — Quel seuil de confiance utiliser en production ?

> **Question opérationnelle** : en usine, on ne peut pas se permettre de fausses alarmes
> (arrêt de chaîne) ni de défauts manqués (rejet client). On cherche un **seuil de confiance**
> au-delà duquel on accepte la décision automatique, et en-dessous duquel on demande une
> **vérification humaine**.

On trace, pour chaque classe, la courbe **précision vs couverture** en faisant varier
le seuil de confiance. On en déduit un seuil opérationnel.


In [ ]:
# Probas du meilleur modèle (réutilise best_probs / best_pred / best_conf du Test #2)
thresholds = np.linspace(0.30, 0.99, 30)

# Pour chaque seuil : précision et couverture global + par classe
global_curve = []
per_class_curves = {cls: [] for cls in range(NUM_CLASSES)}

for thr in thresholds:
    mask_accept = best_conf >= thr
    if mask_accept.sum() == 0:
        global_curve.append((thr, np.nan, 0.0))
        for cls in range(NUM_CLASSES):
            per_class_curves[cls].append((thr, np.nan, 0.0))
        continue

    # Global
    accepted_pred = best_pred[mask_accept]
    accepted_true = y_test[mask_accept]
    global_acc = (accepted_pred == accepted_true).mean()
    coverage = mask_accept.mean()
    global_curve.append((thr, global_acc, coverage))

    # Per-class precision (parmi les prédictions acceptées de cette classe, combien sont correctes)
    for cls in range(NUM_CLASSES):
        cls_pred_mask = (best_pred == cls) & mask_accept
        if cls_pred_mask.sum() == 0:
            per_class_curves[cls].append((thr, np.nan, 0.0))
        else:
            prec = (y_test[cls_pred_mask] == cls).mean()
            cov = cls_pred_mask.sum() / max((y_test == cls).sum(), 1)
            per_class_curves[cls].append((thr, prec, cov))

global_curve = np.array(global_curve)

# ------------------------------------------------------------
# Graphe 1 : précision globale et couverture vs seuil
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(global_curve[:, 0], global_curve[:, 1], "o-", color="#3498DB", label="Précision (acceptés)")
axes[0].plot(global_curve[:, 0], global_curve[:, 2], "s-", color="#E67E22", label="Couverture (% accepté)")
axes[0].set_xlabel("Seuil de confiance"); axes[0].set_ylabel("Score")
axes[0].set_title("Précision vs couverture (global)")
axes[0].legend(); axes[0].grid(alpha=0.4); axes[0].set_ylim(0, 1.05)

# ------------------------------------------------------------
# Graphe 2 : précision par classe vs seuil
# ------------------------------------------------------------
for cls in range(NUM_CLASSES):
    arr = np.array(per_class_curves[cls])
    axes[1].plot(arr[:, 0], arr[:, 1], "o-", color=CLASS_COLORS[cls], label=CLASS_NAMES[cls], alpha=0.8)
axes[1].set_xlabel("Seuil de confiance"); axes[1].set_ylabel("Précision par classe")
axes[1].set_title("Précision par classe selon le seuil")
axes[1].legend(loc="lower right", fontsize=8); axes[1].grid(alpha=0.4); axes[1].set_ylim(0, 1.05)

plt.suptitle("Trade-off précision / couverture pour déploiement industriel", fontweight="bold")
plt.tight_layout(); plt.savefig("diagnostic_test3_thresholds.png", dpi=150, bbox_inches="tight")
plt.show()

# ------------------------------------------------------------
# Choix automatique d'un seuil opérationnel
# ------------------------------------------------------------
# Cible : précision globale >= 95% tout en gardant >=80% de couverture
target_precision = 0.95
target_coverage = 0.80

best_thr = None
for thr, prec, cov in global_curve:
    if np.isnan(prec):
        continue
    if prec >= target_precision and cov >= target_coverage:
        best_thr = thr
        break

print(f"🎯 Objectif : précision ≥ {target_precision:.0%}  ET  couverture ≥ {target_coverage:.0%}")
if best_thr is not None:
    idx = np.argmin(np.abs(global_curve[:, 0] - best_thr))
    print(f"\n✅ Seuil recommandé : {best_thr:.2f}")
    print(f"   → précision automatique = {global_curve[idx, 1]:.2%}")
    print(f"   → couverture automatique = {global_curve[idx, 2]:.2%}")
    print(f"   → {(1 - global_curve[idx, 2])*100:.1f}% des cas envoyés en revue humaine")
else:
    # Fallback : meilleur compromis (max précision*couverture)
    valid = ~np.isnan(global_curve[:, 1])
    score = global_curve[valid, 1] * global_curve[valid, 2]
    best_idx_valid = np.argmax(score)
    valid_thrs = global_curve[valid, 0]
    fallback_thr = valid_thrs[best_idx_valid]
    print(f"\n⚠️  Objectif non atteint. Compromis recommandé : seuil = {fallback_thr:.2f}")
    print(f"   → précision = {global_curve[valid, 1][best_idx_valid]:.2%}")
    print(f"   → couverture = {global_curve[valid, 2][best_idx_valid]:.2%}")

# Précision par classe au seuil choisi
print("\n📋 Précision par classe au seuil retenu (ou compromis) :")
chosen = best_thr if best_thr is not None else fallback_thr
for cls in range(NUM_CLASSES):
    arr = np.array(per_class_curves[cls])
    i = np.argmin(np.abs(arr[:, 0] - chosen))
    p = arr[i, 1]
    c = arr[i, 2]
    p_str = "n/a" if np.isnan(p) else f"{p:.2%}"
    print(f"  {CLASS_NAMES[cls]:<12} précision={p_str}  couverture={c:.2%}")


---
# 18. Comparaison finale


In [ ]:
summary = df_results.copy()
summary["ΔF1_vs_baseline"] = (summary["F1_macro"] - summary.loc[0, "F1_macro"]).round(4)

print("=" * 80)
print(summary.to_string(index=False, float_format="%.4f"))
print("=" * 80)

best_idx = summary["F1_macro"].idxmax()
print(f"\n🏆 Champion : {summary.loc[best_idx, 'Modèle']}")
print(f"   Accuracy  : {summary.loc[best_idx, 'Accuracy']:.4f}")
print(f"   F1 macro  : {summary.loc[best_idx, 'F1_macro']:.4f}")
print(f"   F1 weight : {summary.loc[best_idx, 'F1_weighted']:.4f}")

# Bar chart
fig, ax = plt.subplots(figsize=(11, 5))
x = np.arange(len(summary))
w = 0.35
ax.bar(x - w/2, summary["Accuracy"], w, label="Accuracy", color="#3498DB")
ax.bar(x + w/2, summary["F1_macro"], w, label="F1 macro", color="#E67E22")
ax.set_xticks(x); ax.set_xticklabels(summary["Modèle"], rotation=20, ha="right")
ax.set_ylim(0, 1.05); ax.grid(axis="y", alpha=0.4)
ax.set_title("Comparaison finale — accuracy vs F1 macro", fontweight="bold")
ax.legend()
for i, (a, b) in enumerate(zip(summary["Accuracy"], summary["F1_macro"])):
    ax.text(i - w/2, a + 0.01, f"{a:.3f}", ha="center", fontsize=8)
    ax.text(i + w/2, b + 0.01, f"{b:.3f}", ha="center", fontsize=8)
plt.tight_layout()
plt.savefig("final_comparison.png", dpi=150, bbox_inches="tight")
plt.show()


---
# 19. Conclusion technique

### Pipeline réalisé

1. ✅ Dataset **Magnetic Tile Defect** (hidden gem, 1344 images, 6 classes, masques GT)
2. ✅ Chargement récursif + parsing des masques pixel-précis
3. ✅ Prétraitement grayscale 224×224 + **CLAHE**
4. ✅ Split stratifié **70/15/15** + **class_weight** Sklearn balanced
5. ✅ Augmentation industrielle + **CutMix**
6. ✅ **Focal Loss** (γ=2) + Label Smoothing (0.05) pour le déséquilibre
7. ✅ Baseline CNN (4 blocs) — référence
8. ✅ Improved CNN : **Residual + SE + CBAM** (attention canal + spatial)
9. ✅ **Cosine LR avec warmup**
10. ✅ **EfficientNetV2B0** transfer learning en 2 stages
11. ✅ **TTA** ×10
12. ✅ **Calibration** : ECE + reliability diagram
13. ✅ **Grad-CAM + IoU vs masques GT** (innovation propre à ce dataset)
14. ✅ t-SNE des embeddings
15. ✅ **Robustesse** face au bruit / flou / luminosité

### Améliorations clés vs NEU-DET notebook original

| Apport | Bénéfice |
|---|---|
| **CBAM** au lieu de SE seul | Attention spatiale → meilleure localisation des défauts |
| **CutMix** au lieu de Mixup | Préserve les textures locales (cruciales en défaut industriel) |
| **Focal Loss + class_weight douce** | Gère le déséquilibre 30:1 (impossible sur NEU-DET équilibré) |
| **Warmup + Cosine LR** | Convergence plus stable, surtout en début d'entraînement |
| **IoU Grad-CAM vs masque GT** | Métrique objective d'interprétabilité (unique à ce dataset) |
| **Reliability + ECE** | Mesure la calibration — critique en décision industrielle |
| **Tests de robustesse** | Évaluation conditions réelles d'usage |

### Trois tests diagnostiques décisifs (narratif présentation)

| Test | Constat | Décision technique |
|---|---|---|
| **#1 — Modèle naïf sans correction** | Le CNN naïf s'effondre vers la classe `Free` (F1_macro ~ 0.14) | → On ajoute `class_weight` sqrt-balanced + Focal Loss douce |
| **#2 — Analyse des erreurs** | Confusion concentrée sur 1-2 paires de classes ; le modèle est plus confiant quand il a raison | → On peut filtrer par seuil → voir Test #3 |
| **#3 — Seuil de confiance** | Au-dessus de ~0.85, la précision dépasse 95% sur 80%+ de la couverture | → Recommandation déploiement : auto-classification + revue humaine en dessous du seuil |

### Points d'attention

- La classe **Fray** (32 images) reste la plus difficile : peu d'échantillons, frontière floue avec `Uneven`.
- L'IoU Grad-CAM montre où le modèle regarde **vraiment** — utile pour gagner la confiance des opérateurs.
- En production, prévoir un **recalibrage** (temperature scaling) si l'ECE > 0.05.

> Ce notebook démontre qu'avec les bonnes techniques (focal loss, CBAM, class weights, IoU-grounded explainability),
> un dataset déséquilibré et bruité reste exploitable à un haut niveau de précision tout en restant **interprétable**.
